# 从零实现 Unigram：lattice、Viterbi、EM 与采样

## 学习目标

完成本 notebook 后，你应该能够：

1. 把字符串的所有合法子词切分表示成 lattice；
2. 用 Viterbi 求最大概率路径，并说明它为何不是最长匹配；
3. 用 log-sum-exp 和 forward-backward 计算边缘概率与期望计数；
4. 解释候选初始化、EM、删除损失与迭代剪枝；
5. 区分 n-best、随机 subword regularization 和确定性推理。

> 代码只依赖 Python 标准库，使用很小的手工词表展示数学结构；生产训练器还需高效 Trie、候选生成和大规模并行统计。


## 1. 原理与公式：概率模型与 lattice

对一条切分 $z=(t_1,\ldots,t_k)$，Unigram 假设：

\[
P(z)=\prod_i p(t_i),\qquad \operatorname{cost}(z)=-\sum_i\log p(t_i).
\]

字符串 $x$ 的观测概率需要对所有合法切分求和：

\[
P(x)=\sum_{z\in S(x)}\prod_{t\in z}p(t).
\]

lattice 的节点是字符串位置；若 `text[i:j]` 是词表 piece，就建立一条 $i\to j$ 的边，边权为 $-\log p(t)$。任意从 0 到末尾的路径就是一种完整切分。


In [ ]:
import math
import random
from collections import Counter

# 权重归一化为合法的 unigram 概率。字符项保证示例字符串可达。
weights = {
    "a": 40.0, "b": 20.0, "c": 25.0,
    "ab": 5.0, "bc": 30.0, "abc": 0.5,
    "u": 5.0, "n": 5.0, "i": 20.0, "g": 10.0,
    "r": 10.0, "m": 10.0, "un": 20.0,
    "uni": 50.0, "gram": 50.0,
}
normalizer = sum(weights.values())
probabilities = {token: weight / normalizer for token, weight in weights.items()}
log_probs = {token: math.log(probability) for token, probability in probabilities.items()}

def build_lattice(text, model_log_probs):
    edges = [[] for _ in range(len(text) + 1)]
    for start in range(len(text)):
        for token in sorted(model_log_probs):
            if text.startswith(token, start):
                end = start + len(token)
                edges[start].append((end, token, model_log_probs[token]))
    return edges

def show_lattice(text, edges):
    for start, outgoing in enumerate(edges[:-1]):
        rendered = [(end, token, round(logp, 3)) for end, token, logp in outgoing]
        print(f"位置 {start}: {rendered}")

text = "abc"
lattice = build_lattice(text, log_probs)
show_lattice(text, lattice)


## 2. Viterbi：求最佳完整路径

令 `dp[j]` 为覆盖前 `j` 个字符的最小负 log 代价：

\[
dp[j]=\min_{(i,j,t)}\{dp[i]-\log p(t)\},\quad dp[0]=0.
\]

每次更新终点时保存 `(前驱位置, token)`，最后从字符串末尾回溯。复杂度是 $O(E)$，其中 $E$ 是 lattice 边数。真实词表应用 Trie 枚举从每个起点出发的候选，而不是检查所有 token。


In [ ]:
def viterbi(text, model_log_probs):
    edges = build_lattice(text, model_log_probs)
    dp = [math.inf] * (len(text) + 1)
    back = [None] * (len(text) + 1)
    dp[0] = 0.0
    for start in range(len(text)):
        if math.isinf(dp[start]):
            continue
        for end, token, logp in edges[start]:
            candidate = dp[start] - logp
            if candidate < dp[end]:
                dp[end] = candidate
                back[end] = (start, token)
    if back[-1] is None and text:
        raise ValueError(f"输入 {text!r} 没有完整切分路径")
    pieces = []
    position = len(text)
    while position > 0:
        previous, token = back[position]
        pieces.append(token)
        position = previous
    return dp[-1], list(reversed(pieces))

for sample in ["abc", "unigram"]:
    cost, pieces = viterbi(sample, log_probs)
    print(f"{sample!r:10} -> {pieces}, cost={cost:.4f}")

# abc 在词表中，但概率很低；最佳路径不一定选择最长 token。
assert viterbi("abc", log_probs)[1] != ["abc"]


## 3. Forward-backward：对所有路径求和

Viterbi 使用 `min/max` 只保留一条路径；Unigram 训练需要所有路径的边缘概率。直接连乘会下溢，因此在 log 域使用：

\[
\operatorname{LSE}(v_1,\ldots,v_k)=m+\log\sum_i e^{v_i-m},\quad m=\max_i v_i.
\]

前向值汇总到达位置的所有路径，后向值汇总从位置到终点的所有路径。边 $(i,j,t)$ 的 posterior 为：

\[
\exp(\alpha_i+\log p(t)+\beta_j-\log P(x)).
\]

把相同 token 的边 posterior 相加，就是 EM 的 E 步期望计数。


In [ ]:
NEG_INF = float("-inf")

def log_add(left, right):
    if left == NEG_INF:
        return right
    if right == NEG_INF:
        return left
    maximum = max(left, right)
    return maximum + math.log(math.exp(left - maximum) + math.exp(right - maximum))

def forward_backward(text, model_log_probs):
    edges = build_lattice(text, model_log_probs)
    size = len(text) + 1
    alpha = [NEG_INF] * size
    alpha[0] = 0.0
    for start in range(len(text)):
        for end, token, logp in edges[start]:
            alpha[end] = log_add(alpha[end], alpha[start] + logp)

    beta = [NEG_INF] * size
    beta[-1] = 0.0
    for start in range(len(text) - 1, -1, -1):
        for end, token, logp in edges[start]:
            beta[start] = log_add(beta[start], logp + beta[end])

    log_z = alpha[-1]
    if log_z == NEG_INF:
        raise ValueError(f"输入 {text!r} 没有完整切分路径")
    expected = Counter()
    for start in range(len(text)):
        for end, token, logp in edges[start]:
            posterior = math.exp(alpha[start] + logp + beta[end] - log_z)
            expected[token] += posterior
    return log_z, expected, alpha, beta

log_z, expected, alpha, beta = forward_backward("abc", log_probs)
print("log P('abc') =", round(log_z, 6))
print("期望 token 次数 =", {k: round(v, 4) for k, v in expected.items()})
assert abs(alpha[-1] - beta[0]) < 1e-12


## 4. EM 与从大到小剪枝

固定候选词表时，EM 反复执行：

1. **E 步**：对全部句子做 forward-backward，汇总每个 token 的期望次数 $c_t$；
2. **M 步**：更新 $p(t)=c_t/\sum_u c_u$；
3. 收敛后估计删除每个候选的损失，优先删除有低损失替代路径的 token；
4. 保留 required chars、特殊 token 和 byte fallback，再在缩小词表上重新 EM。

仅按最低概率删除是错误的：一个罕见字符概率虽低，却可能是某些输入的唯一覆盖。批量剪枝太激进也可能同时删掉互为替代的 pieces。


In [ ]:
def best_cost_without(text, model_log_probs, removed_token):
    reduced = {token: score for token, score in model_log_probs.items() if token != removed_token}
    try:
        return viterbi(text, reduced)[0]
    except ValueError:
        return math.inf

base_cost, _ = viterbi("abc", log_probs)
for token in ["abc", "bc", "a"]:
    new_cost = best_cost_without("abc", log_probs, token)
    increase = new_cost - base_cost
    print(f"删除 {token!r:5}: 最佳路径代价增量 = {increase}")

print("注意：这是单句 Viterbi 的教学近似；生产剪枝会在全语料概率目标上估计删除损失。")


## 5. N-best 与 subword regularization

Viterbi 返回一条最佳路径；n-best 返回分数最高的前 N 条；subword regularization 则从路径分布采样。常见温度化形式为：

\[
q_\alpha(z\mid x)=\frac{P(z)^\alpha}{\sum_{z'}P(z')^\alpha}.
\]

`alpha` 越大越集中于最佳路径，越小越平坦。训练时随机切分是一种数据增强，推理通常关闭以保证确定性和缓存命中。下面仅对短字符串穷举路径，便于验证；生产实现应在 lattice 上做 k-shortest paths 或后向采样。


In [ ]:
def enumerate_paths(text, model_log_probs):
    edges = build_lattice(text, model_log_probs)
    paths = []
    def visit(position, tokens, log_score):
        if position == len(text):
            paths.append((log_score, list(tokens)))
            return
        for end, token, logp in edges[position]:
            tokens.append(token)
            visit(end, tokens, log_score + logp)
            tokens.pop()
    visit(0, [], 0.0)
    return sorted(paths, key=lambda item: (-item[0], item[1]))

paths = enumerate_paths("abc", log_probs)
print("全部路径（按概率从高到低）:")
for log_score, path in paths:
    print(path, "prob=", round(math.exp(log_score), 7))

def sample_from_paths(paths, alpha=1.0, seed=42):
    rng = random.Random(seed)
    scaled = [alpha * score for score, _ in paths]
    maximum = max(scaled)
    weights = [math.exp(score - maximum) for score in scaled]
    return rng.choices([path for _, path in paths], weights=weights, k=1)[0]

print("固定随机种子的采样结果:", sample_from_paths(paths, alpha=0.7, seed=7))


## 6. 边界、误区与相邻算法对比

- **数值稳定**：forward/backward 用 log-sum-exp，Viterbi 才用 max/min；
- **基础覆盖**：required chars 或 byte fallback 必须保证终点可达；
- **零长度 token**：会产生自环，必须禁止；
- **剪枝后要重新 EM**：旧概率分布不再归一；
- **SentencePiece 不等于 Unigram**：它是可承载 Unigram、BPE 等算法的工具框架；
- **piece score 不是 LLM 置信度**：它只服务于表面字符串切分。

| 方法 | 训练方向 | 推理选择 | 随机切分 |
|---|---|---|---|
| Unigram | 大候选 → EM → 删除低损失项 | 全局概率 Viterbi | 天然支持 |
| BPE | 小底座 → 高频 pair merge | 固定 merge rank | 需 BPE-dropout 等扩展 |
| WordPiece | 候选收益扩词表 | 局部最长匹配 | 标准版本无 |


## 练习与面试总结

1. 用 Trie 替换 `build_lattice` 中的全词表扫描。
2. 在一个小语料上实现完整 EM，验证每轮语料似然不下降。
3. 为每个候选估计全语料删除损失，并保护 required chars。
4. 实现无需枚举全部路径的后向采样，比较不同 `alpha` 的长度分布。
5. 加入空格元符号和 byte fallback，并测试严格 round-trip。

**一分钟回答**：Unigram 给每个 piece 一个概率，把所有合法子词放进位置 lattice。确定性编码用 Viterbi 找负 log 代价最小的完整路径；训练把切分当隐变量，用 forward-backward/EM 估计期望计数，再从大候选词表中删除替代损失小的项。它比最长匹配更全局，并天然支持 n-best 与采样，但需要处理候选规模、数值稳定、基础覆盖、剪枝近似和完整模型制品兼容。
